# Phase 5 — Temporal Aggregation Features

Derive per-well summary features from the production panel: rate-based,
pressure-based, field-state, and volatility features. Save as well_features.parquet.

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
elif 'mari_poc' not in os.getcwd(): os.chdir('mari_poc')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.config import PROCESSED_DIR, FIGURES_DIR
from src.features import (
    compute_wgr, detect_breakthrough,
    compute_rate_features, compute_pressure_features,
    compute_field_state_features, compute_volatility_features
)

panel = pd.read_parquet(PROCESSED_DIR / 'panel_long.parquet')
static = pd.read_parquet(PROCESSED_DIR / 'well_static.parquet')

panel_wgr = compute_wgr(panel)
bt_df = detect_breakthrough(panel_wgr)

print(f'Panel: {panel.shape}, Static: {static.shape}')

Panel: (6865, 9), Static: (22, 15)


In [2]:
# Compute all temporal feature groups
rate_feats = compute_rate_features(panel, static)
print(f'Rate features: {rate_feats.shape}')
print(rate_feats.head())
print(f'Nulls:\n{rate_feats.isnull().sum()}')

Rate features: (22, 10)
         well  mean_gas_yr1  mean_gas_yr2  mean_gas_yr3  cum_gas_yr1  \
0    M-11-HRL    177.547083     83.675000      9.658833     2130.565   
1  M-122H-HRL    300.017083    252.185333    222.849429     3600.205   
2  M-123H-HRL    174.038333    232.107250           NaN     2088.460   
3  M-124H-HRL    203.580364    421.959714           NaN     2239.384   
4  M-125H-HRL    257.674300           NaN           NaN     2576.743   

   cum_gas_yr2  cum_gas_yr5  peak_gas  time_to_peak_months   arps_di  
0     3134.665     5813.755   321.709                    1  0.007721  
1     6626.429     8186.375   409.398                    1  0.017610  
2     3945.318     3945.318   300.044                    2       NaN  
3     5193.102     5193.102   475.617                   15       NaN  
4     2576.743     2576.743   333.707                    4       NaN  
Nulls:
well                   0
mean_gas_yr1           0
mean_gas_yr2           2
mean_gas_yr3           4
cum_gas_yr

In [3]:
pressure_feats = compute_pressure_features(panel, static, bt_df)
print(f'Pressure features: {pressure_feats.shape}')
print(pressure_feats.head())
print(f'Nulls:\n{pressure_feats.isnull().sum()}')

Pressure features: (22, 6)
         well  initial_whfp  whfp_decline_rate  whfp_at_bt  drawdown_proxy  \
0    M-11-HRL           NaN          19.635974       508.5             NaN   
1  M-122H-HRL    552.050000          64.158637       520.4          114.21   
2  M-123H-HRL    440.866667          53.021061       391.1             NaN   
3  M-124H-HRL    603.700000         161.775806       354.8             NaN   
4  M-125H-HRL    485.500000         169.460000       385.8             NaN   

    whfp_std  
0        NaN  
1  42.517420  
2  45.920817  
3  88.311639  
4        NaN  
Nulls:
well                  0
initial_whfp         12
whfp_decline_rate     0
whfp_at_bt            0
drawdown_proxy       16
whfp_std              8
dtype: int64


In [4]:
field_state_feats = compute_field_state_features(panel, static)
print(f'Field state features: {field_state_feats.shape}')
print(field_state_feats.head())
print(f'Nulls:\n{field_state_feats.isnull().sum()}')

Field state features: (22, 4)
         well  cum_field_gas_at_spud  cum_field_water_at_spud  \
0    M-11-HRL                  0.000                    0.000   
1  M-122H-HRL             702078.903              2169985.184   
2  M-123H-HRL             723509.996              2379399.759   
3  M-124H-HRL             725871.303              2400934.664   
4  M-125H-HRL             747452.464              2649136.220   

   active_wells_at_spud  
0                     0  
1                    17  
2                    18  
3                    19  
4                    20  
Nulls:
well                       0
cum_field_gas_at_spud      0
cum_field_water_at_spud    0
active_wells_at_spud       0
dtype: int64


In [5]:
volatility_feats = compute_volatility_features(panel, static)
print(f'Volatility features: {volatility_feats.shape}')
print(volatility_feats.head())
print(f'Nulls:\n{volatility_feats.isnull().sum()}')

Volatility features: (22, 2)
         well  gas_cov_2yr
0    M-11-HRL     0.776036
1  M-122H-HRL     0.205689
2  M-123H-HRL     0.522964
3  M-124H-HRL     0.431067
4  M-125H-HRL     0.313039
Nulls:
well           0
gas_cov_2yr    0
dtype: int64


In [ ]:
# Merge all features onto static
from src.features import compute_d_to_gwc

well_features = static.copy()
for feat_df in [rate_feats, pressure_feats, field_state_feats, volatility_feats]:
    well_features = well_features.merge(feat_df, on='well', how='left')

# Add breakthrough info
well_features = well_features.merge(bt_df, on='well', how='left')

# Add d_to_gwc (distance from bottom perf to GWC — verticals only)
d_gwc = compute_d_to_gwc(static)
well_features = well_features.merge(d_gwc, on='well', how='left')

print(f'\nFull feature table: {well_features.shape}')
print(f'Columns: {list(well_features.columns)}')
print(f'\nNull summary:')
print(well_features.isnull().sum())

# Save
well_features.to_parquet(PROCESSED_DIR / 'well_features.parquet', index=False)
print(f'\nSaved well_features.parquet')

In [7]:
# Display full feature table
print(well_features.to_string())

          well  porosity  permeability_md  skin    sw   net_pay_m  chlorides_ppm  top_perf_md  bottom_perf_md  gas_gravity  is_horizontal first_prod_date last_prod_date  production_months  rkb_elevation_m  mean_gas_yr1  mean_gas_yr2  mean_gas_yr3  cum_gas_yr1  cum_gas_yr2  cum_gas_yr5  peak_gas  time_to_peak_months   arps_di  initial_whfp  whfp_decline_rate  whfp_at_bt  drawdown_proxy    whfp_std  cum_field_gas_at_spud  cum_field_water_at_spud  active_wells_at_spud  gas_cov_2yr  bt_detected    bt_date bt_month_index
0     M-11-HRL      0.20              7.4  -0.1  0.45   10.500000          17000   695.000000      705.500000         0.70          False      1978-03-01     2025-06-01                568             70.0    177.547083     83.675000      9.658833     2130.565     3134.665     5813.755   321.709                    1  0.007721           NaN          19.635974       508.5             NaN         NaN                  0.000                    0.000                     0     0.77

## Findings

1. Rate features computed for all 22 wells: mean gas years 1-3, cumulative gas years 1/2/5, peak gas, time to peak, Arps Di
2. Pressure features: initial WHFP, decline rate, WHFP at BT, drawdown proxy, WHFP std — some nulls for wells with late WHFP coverage
3. Field-state features capture depletion vintage: older wells had less cumulative field production at spud, newer wells see a depleted reservoir
4. Arps Di (exponential decline rate): computed from months 24-120 where available; reflects reservoir drive character
5. gas_cov_2yr captures early-life rate volatility — may indicate unstable flow or operational intermittency
6. Horizontal wells have higher initial rates and peak rates than verticals (expected)
7. cum_field_gas_at_spud is a strong vintage proxy — monotonically increasing with time
8. drawdown_proxy captures early pressure depletion — relevant for coning behavior
9. d_to_gwc_m: distance from bottom perforation to GWC (verticals only; NaN for horizontals where MD ≠ TVD)
10. Well_features.parquet is the complete feature table for Phase 6 modeling
11. Nulls in pressure features for oldest wells (pre-1990 first production, WHFP data starts 1990)